### Model for predicting the outperformer

In [17]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_parquet("game.parquet")
print(df.head())

                    channel  subscribers_cc_x  category_cc_x  \
0  UC-lHJZR3Gqxm24_Vd_AJ5Yw         101000000         Gaming   
1  UCbCmjCuTUZos6Inko4u57UQ          60100000      Education   
2  UCpEhnqL0y41EpW2TvWAHD7Q          56018869  Entertainment   
3  UC295-Dw_tDNtZXFeAPAW6Aw          60600000  Howto & Style   
4  UCJ5v_MCY6GNUBTO8-D3XoAg          48400000         Sports   

   channel_age_days_x  videos_per_day_x  avg_engagement_efficiency_x  \
0              3468.0          1.140715                     0.041988   
1              4804.0          0.095337                     0.002954   
2              4785.0          6.825705                     0.004994   
3              1076.0          3.337361                     0.011340   
4              4552.0          9.538884                     0.015842   

   total_views_x  total_like_count_x  total_num_comms_x  \
0   2.345148e+10         950538161.0         72904068.0   
1   4.323417e+10          96951076.0           611177.0   
2   4

In [44]:
df.columns

Index(['channel', 'subscribers_cc_x', 'category_cc_x', 'channel_age_days_x',
       'videos_per_day_x', 'avg_engagement_efficiency_x', 'total_views_x',
       'total_like_count_x', 'total_num_comms_x', 'video_count_processed_x',
       'weights_x', 'E_avg', 'E_sums', 'logE', 'logS', 'r_c', 'category_cc_y',
       'join_date', 'name_cc', 'subscribers_cc_y', 'videos_cc',
       'subscriber_rank_sb', 'weights_y', 'channel_age_days_y',
       'videos_per_day_y', 'avg_engagement_efficiency_y', 'total_views_y',
       'total_duration', 'total_like_count_y', 'total_dislike_count',
       'total_num_comms_y', 'avg_video_like_ratio', 'video_count_processed_y',
       'is_outperformer'],
      dtype='object')

In [45]:
keep_cols = [
    # context / structural
    "category_cc_x",
    "channel_age_days_x",
    "videos_per_day_x",

    # duration components (for later transform)
    "total_duration",
    "video_count_processed_x",

    # target
    "is_outperformer",
]

df_game = df[keep_cols].copy()
df_game

,category_cc_x,channel_age_days_x,videos_per_day_x,total_duration,video_count_processed_x,is_outperformer
0,Gaming,3468.0,1.140715,3449869.0,3927,False
1,Education,4804.0,0.095337,519007.0,462,False
2,Entertainment,4785.0,6.825705,30501034.0,23903,False
3,Howto & Style,1076.0,3.337361,3383229.0,3624,False
4,Sports,4552.0,9.538884,7868933.0,40549,False
...,...,...,...,...,...,...
136465,Music,1116.0,0.094086,18229.0,105,False
136466,People & Blogs,2201.0,0.159927,176799.0,352,False
136467,Gaming,1633.0,0.290876,210251.0,475,False
136468,Music,3119.0,0.004809,3016.0,15,False


In [46]:
df_game["avg_duration"] = (df_game["total_duration"] / df_game["video_count_processed_x"])
df_game["log_duration"] = np.log(df_game["avg_duration"])
df_game["log_channel_age_days"] = np.log(df_game["channel_age_days_x"])

In [47]:
df_game.describe()

df_game.columns
df_game["category"] = df_game["category_cc_x"].copy()
keep_cols_final = ["category", "log_duration", "log_channel_age_days", "is_outperformer", "videos_per_day_x"]
df_game = df_game[keep_cols_final].copy()
df_game.columns

Index(['category', 'log_duration', 'log_channel_age_days', 'is_outperformer',
       'videos_per_day_x'],
      dtype='object')

In [58]:
# Standardise the variables

NUM_COLS = [
    "videos_per_day_x",
    "log_duration",
    "log_channel_age_days",
]
for col in NUM_COLS:
    df_game[col] = (df_game[col] - df_game[col].mean()) / df_game[col].std()
    df_game[col] = pd.to_numeric(df_game[col])
df_game["category"] = df_game["category"].astype(str)
df_game["is_outperformer"] = df_game["is_outperformer"].astype(int)
print(df_game.info())
df_game

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136470 entries, 0 to 136469
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   category              136470 non-null  object 
 1   log_duration          136470 non-null  float64
 2   log_channel_age_days  136469 non-null  float64
 3   is_outperformer       136470 non-null  int64  
 4   videos_per_day_x      136470 non-null  float64
dtypes: float64(3), int64(1), object(1)
memory usage: 5.2+ MB
None


,category,log_duration,log_channel_age_days,is_outperformer,videos_per_day_x
0,Gaming,0.793819,0.869592,0,0.396541
1,Education,1.072170,1.389630,0,-0.106549
2,Entertainment,1.216392,1.383306,0,3.132454
3,Howto & Style,0.862635,-0.998064,0,1.453681
4,Sports,-0.915565,1.303643,0,4.438176
...,...,...,...,...,...
136465,Music,-1.041623,-0.939815,0,-0.107151
136466,People & Blogs,0.160939,0.144017,0,-0.075464
136467,Gaming,0.017859,-0.332330,0,-0.012445
136468,Music,-0.875413,0.700328,0,-0.150115


In [59]:
from statsmodels.formula.api import logit

mod = logit(formula='is_outperformer ~ videos_per_day_x + log_duration + log_channel_age_days + C(category)', data=df_game)

res = mod.fit()
print(res.summary())

Optimization terminated successfully.
         Current function value: 0.094192
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:        is_outperformer   No. Observations:               136469
Model:                          Logit   Df Residuals:                   136450
Method:                           MLE   Df Model:                           18
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                 0.06669
Time:                        11:29:51   Log-Likelihood:                -12854.
converged:                       True   LL-Null:                       -13773.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                               -4.9799      0.

In [60]:
params = res.params
params

Intercept                              -4.979949
C(category)[T.Comedy]                   1.324946
C(category)[T.Education]                0.435543
C(category)[T.Entertainment]            1.001750
C(category)[T.Film and Animation]       0.786877
C(category)[T.Gaming]                   1.621890
C(category)[T.Howto & Style]            1.084877
C(category)[T.Music]                   -0.455019
C(category)[T.News & Politics]          1.934682
C(category)[T.None]                     0.340292
C(category)[T.Nonprofits & Activism]    0.822906
C(category)[T.People & Blogs]           1.258099
C(category)[T.Pets & Animals]           0.911945
C(category)[T.Science & Technology]     0.416626
C(category)[T.Sports]                  -0.544879
C(category)[T.Travel & Events]          0.587978
videos_per_day_x                       -0.425750
log_duration                            0.167925
log_channel_age_days                   -0.399734
dtype: float64

In [63]:
import json

coef = {
    "intercept": params["Intercept"],
    "videos_per_day_x": params["videos_per_day_x"],
    "log_duration": params["log_duration"],
    "log_channel_age_days": params["log_channel_age_days"],
}

category_coef = {}

for name, value in params.items():
    if name.startswith("C(category)[T."):
        cat = name.split("T.")[1].rstrip("]")
        category_coef[cat] = value
        
game_model = {
    "coef": coef,
    "category_coef": category_coef,
    "reference_category": "BASE",  # implicit statsmodels baseline
}

with open("logit_game_model.json", "w") as f:
    json.dump(game_model, f, indent=2)